# PyperCache

Welcome to PyperCache! This library provides **persistent, file-backed caching** for Python applications, with pluggable storage backends and powerful query capabilities. Whether you're caching API responses, storing computation results, or building audit trails, PyperCache offers a robust foundation.

## What You'll Learn

By the end of this notebook, you'll understand:
- **Core caching operations**: Store, retrieve, and manage cached data with TTL and staleness semantics
- **Storage backends**: Choose the right backend (.pkl, .json, .manifest, .db) for your use case
- **Data querying**: Navigate cached JSON-like structures with safe, read-only queries
- **Type safety**: Round-trip objects with automatic serialization/deserialization
- **Audit logging**: Track requests with append-only JSON Lines logging
- **Performance considerations**: When and how to use each feature effectively

Let's dive in and explore PyperCache's capabilities through hands-on examples!

## Setup

Prepare the environment for the demo by importing necessary libraries and ensuring the PyperCache package is accessible.

In [ ]:
import sys, os, tempfile, time, json, threading, shutil
from pathlib import Path

# Ensure PyperCache package is importable from the repo root
repo_root = Path.cwd()
if not (repo_root / "pypercache").exists():
    repo_root = repo_root.parent

sys.path.insert(0, str(repo_root))

print("Python", sys.version.split()[0])
print("Working directory:", Path.cwd())

## Package Structure

PyperCache is structured as a layered package with clear dependency boundaries:
- `utils` contains general-purpose utilities.
- `core` implements the main caching logic.
- `storage` provides pluggable backends.
- `query` enables read-only navigation of cached data.

In [ ]:
from pypercache import Cache, RequestLogger
import pypercache

# Show package metadata
print(f"pypercache version: {pypercache.__version__}")

# Demonstrate that the primary API is accessible at the top level
print(f"\nMain entry points:")
print(f"  - {Cache.__module__}.{Cache.__name__}")
print(f"  - {RequestLogger.__module__}.{RequestLogger.__name__}")

# Sub-packages provide the underlying implementation layers
import pypercache.storage as storage
import pypercache.query as query

print(f"\nArchitecture Layers:")
print(f"  - {storage.__name__:<18} | Pluggable backends (Pickle, JSON, SQLite, etc.)")
print(f"  - {query.__name__:<18} | Read-only navigation (JsonInjester)")

## Part 1: Core Caching Operations

Let's start with the fundamentals. PyperCache's `Cache` class handles storing and retrieving data with automatic persistence to disk.

### Storage Backends

PyperCache supports multiple storage backends, each selected automatically by file extension. This means you can switch backends without changing your code — just change the filename!

### Backend Comparison

| Extension | Backend | Best For | Trade-offs |
|-----------|---------|----------|------------|
| `*.pkl` | Pickle | Speed, Python objects | Not human-readable, Python-only |
| `*.json` | JSON | Debugging, portability | Slower, no binary data |
| `*.manifest` | Chunked | Large caches (100MB+) | More complex, multiple files |
| `*.db` | SQLite | Concurrency | Database overhead |

### Store and Retrieve

Imagine you're building an application that fetches user profiles from an external API. Instead of hitting the API every time, you can cache the responses for faster access and reduced API calls.

In [ ]:
from pypercache import Cache

# Basic store / retrieve
tmp = tempfile.mkdtemp()
CACHE_FILE = os.path.join(tmp, "demo.pkl")

# Creates cache file on disk and any directories if they don't exist.
cache = Cache(filepath=CACHE_FILE)

cache.store("user:1", {"name": "Alice", "role": "admin"})
cache.store("user:2", {"name": "Bob",   "role": "user"})
cache.store("user:3", {"name": "Charlie", "role": "user"})

record = cache.get("user:1")
print("Record data  :", record.data)
print("Is stale     :", record.is_data_stale)
print("Expiry       :", record.expiry)   # math.inf by default

### Understanding TTL and Staleness

PyperCache distinguishes between **TTL (Time-To-Live)** and **staleness** for flexible cache expiration:

- **TTL**: Hard expiration time — data is considered invalid after this point
- **Staleness**: Soft expiration — data might still be acceptable if it's "not too old"

This is useful for APIs where you prefer fresh data but can tolerate slightly stale responses during high load.

In [ ]:
# TTL — store with a 1-second expiry
cache.store("session:abc", {"token": "xyz"}, expiry=1)
print("Fresh right away:", cache.is_data_fresh("session:abc"))
time.sleep(1)
print("Fresh after 1 s: ", cache.is_data_fresh("session:abc"))


### Model Deserialization & Type Casting

PyperCache supports **automatic object hydration** from cached JSON-like data. The `@Cache.cached` decorator registers a class with the cache, enabling type-safe round-trips: store objects as plain dicts, retrieve them as fully reconstructed instances.

**Use case:** When you need Python objects instead of raw dicts during retrieval.  
**Trade-off:** Light reflection overhead; register only the classes you plan to deserialize frequently.

In [ ]:
# Type casting with @Cache.cached
@Cache.cached
class UserProfile:
    def __init__(self, data: dict):
        self.name = data["name"]
        self.role = data["role"]
    def __repr__(self):
        return f"<UserProfile name={self.name!r} role={self.role!r}>"

cache.store("profile:1", {"name": "Alice", "role": "admin"}, cast=UserProfile)
obj = cache.get_object("profile:1")
print("cast object:", obj)
print("isinstance check:", isinstance(obj, UserProfile))

### Simple API Models with `@apimodel`

For small annotation-driven models the `@apimodel` decorator provides a lightweight alternative to `@Cache.cached`. It injects a constructor that accepts a raw dict and adds `from_dict`/`as_dict` helpers. The example below shows storing a user with `cast=User` and retrieving a typed instance.

In [ ]:
from pypercache import Cache
from pypercache.models.apimodel import apimodel
import tempfile, os, shutil

'''
This demonstrates how to use @apimodel to define nested data structures that can be automatically cast when retrieved from the cache.
In this way, you could quickly create api wrappers with cache and TTL without ever touching JSON responses.
'''

@apimodel
class Address:
    city: str

@apimodel
class Company:
    name: str
    address: Address

@apimodel
class User:
    id: int
    company: Company
    previous: list[Address]

tmp = tempfile.mkdtemp()
cache = Cache(filepath=os.path.join(tmp, 'users_demo.pkl'))

raw = {
    'id': 1,
    'company': {'name': 'Microsoft', 'address': {'city': 'Redmond'}},
    'previous': [{'city': 'Phoenix'}, {'city': 'Tempe'}]
}

cache.store('u', raw, cast=User)
u: User = cache.get_object('u')

print(type(u.company))          # Company
print(type(u.previous[0]))      # Address
print(u.company.address.city)   # Redmond
print(u.as_dict())

shutil.rmtree(tmp)

### Update and Erase

Cache entries can be updated in-place or wiped entirely. This is useful for invalidation patterns — for example, when a user changes their profile, you update the cached entry rather than waiting for TTL expiry.

In [ ]:
# Update and erase
cache.store("profile:1", {"name": "Alice", "role": "admin"})
cache.update("profile:1", {"name": "Alice (updated)", "role": "superadmin"})
print("After update:", cache.get("profile:1").data)

cache.completely_erase_cache()
print("After erase — has profile:1:", cache.has("profile:1"))

shutil.rmtree(tmp)

## Part 2: Querying Cached Data with `JsonInjester`

PyperCache's `JsonInjester` provides safe, read-only navigation over cached JSON-like structures.

### Query Syntax

- **Dotted paths**: Navigate nested objects/arrays
- **Existence filters**: `?active` — find items with a key
- **Value filters**: `?role=admin` — find items where key equals value
- **Pluck operations**: `?name*` — extract all values for a key
- **Defaults**: Handle missing data gracefully

This is perfect for API responses where you need to filter or transform data on retrieval.

### Setup: Cache the Sample Data

Imagine you're building an e-commerce dashboard. You cache order data and need to extract specific information. Let's start by caching a sample order:

In [ ]:
from pypercache import Cache
import tempfile
import os

tmp_dir = tempfile.mkdtemp()
cache = Cache(filepath=os.path.join(tmp_dir, "query_demo.pkl"))

# Store a sample order (simplified for learning)
order_data = {
    "id": "ORD-2024-001",
    "status": "confirmed",
    "customer": {
        "name": "Sarah Johnson",
        "email": "sarah@example.com"
    },
    "items": [
        {"name": "Laptop", "price": 999.99, "category": "electronics"},
        {"name": "Mouse", "price": 59.99, "category": "electronics"},
        {"name": "Case", "price": 29.99, "category": "accessories"}
    ],
    "total": 1089.97
}

cache.store("order:sample", order_data)

# Get the query interface
record = cache.get("order:sample")
query = record.query

print("📦 Sample order cached successfully")
print("=" * 40)

### Basic Navigation: Dotted Paths

The simplest queries use dotted paths to navigate nested structures. Think of it like file system paths, but for JSON data.

In [ ]:
# Navigate to nested properties
customer_name = query.get("customer.name")
order_total = query.get("total")
order_id = query.get("id")

print(f"Customer: {customer_name}")
print(f"Order total: ${order_total}")
print(f"Order ID: {order_id}")

### Working with Arrays: Pluck Operations

When you have a list of items, you often want to extract specific fields from each item. The pluck operator `?fieldname*` does exactly this.

In [ ]:
# Extract all item names from the items array
item_names = query.get("items?name*")
item_prices = query.get("items?price*")

print(f"Item names: {item_names}")
print(f"Item prices: {item_prices}")

# Calculate total from extracted prices
calculated_total = sum(item_prices)
print(f"Calculated total: ${calculated_total}")

### Filtering Data: Value Filters

Need to find items matching specific criteria? Use value filters like `?category=electronics` to get only matching items.

In [ ]:
# Find all electronics
electronics = query.get("items?category=electronics")
print(f"Electronics: {[item['name'] for item in electronics]}")

# Find expensive items (combine with Python logic)
all_items = query.get("items")
expensive = [item for item in all_items if item['price'] > 100]
print(f"Expensive items: {[item['name'] for item in expensive]}")

### Safe Access: Handling Missing Data

APIs don't always return consistent data. Use `default_value` to handle missing fields gracefully.

In [ ]:
# Safe access with defaults
discount = query.get("discount", default_value=0)
shipping_cost = query.get("shipping", default_value=0)
tax_rate = query.get("tax_rate", default_value=0.08)  # 8% default tax

print(f"Discount: ${discount}")
print(f"Shipping: ${shipping_cost}")
print(f"Tax rate: {tax_rate * 100}%")

# Check if fields exist
has_discount = query.get("discount") is not None
print(f"Has discount field: {has_discount}")

### Putting It Together: Building a Dashboard

Now let's combine all the query techniques to extract data for a sales dashboard in a single pass.

In [ ]:
# Dashboard data extraction
dashboard_data = {
    "customer": query.get("customer.name"),
    "order_value": query.get("total"),
    "item_count": len(query.get("items")),
    "categories": list(set(query.get("items?category*"))),  # Unique categories
    "avg_price": sum(query.get("items?price*")) / len(query.get("items"))
}

print("📊 Sales Dashboard Data:")
for key, value in dashboard_data.items():
    print(f"  {key}: {value}")

# Clean up
import shutil
shutil.rmtree(tmp_dir)

## Appendix: Internal Utilities

### The `UNSET` Sentinel

The `UNSET` singleton is used internally to represent "no value" in PyperCache. It ensures consistent behavior across the codebase and avoids ambiguity with `None` or other falsy values. You'll encounter it if you extend PyperCache or inspect returned records directly.

In [ ]:
from pypercache.utils.sentinel import UNSET as U1
from pypercache.utils import UNSET as U2
from pypercache.core.cache import UNSET as U3
from pypercache.utils import UNSET

print("All three import paths return the same object:", U1 is U2 is U3)
print("UNSET is falsy:    ", not UNSET)
print("UNSET is not None: ", UNSET is not None)
print("repr(UNSET):       ", repr(UNSET))